In [6]:
from nltk.corpus import stopwords

#stopwords.words('english')

In [7]:
from sqlalchemy import create_engine
import pandas as pd
import re
import string
import nltk
pd.set_option('display.max_colwidth', 100)

In [8]:
QUERY=""" 
with business_categories as (
select  bc.business_detail_id
,listagg(c.name,'|') within group (order by c.name) as categories
from mysql_oneflare_production.business_categories bc 
left join mysql_oneflare_production.categories c on bc.category_id = c.id
group by 1
)

SELECT
trim(nvl(bm.name,'') || ' ' || nvl(description,'')) "feature"
,bc.categories
,category_name "label"
FROM oneflare_reports.business_master bm
left join business_categories bc on bm.id = bc.business_detail_id
WHERE bm.total_quotes_made >= 3 and category_name is not null
"""

In [9]:
def make_engine():
    user = 'xxx' # change it to your username
    pw = 'xxx' #change it to your password
    db_name = 'snowplow'
    host = 'redshift.oneflare.io'
    port = 5439
    engine_string = "postgresql://%s:%s@%s:%s/%s" % (
        user, pw, host, port, db_name)
    engine = create_engine(engine_string)
    return engine

redshift = make_engine()
data=pd.read_sql(QUERY,redshift)


In [11]:
data.head()

,feature,categories,label
0,"S.F.D Caringbah We are on time and on budget.\nWe have experience in new homes, renovations and ...",Bathroom Renovations|Builder|Carpenter|Decking|Gazebo|Home Renovation|Kitchen Renovations|Pergola,Handyman
1,Gardeners@Work I specialise in all types of fencing and gates.\nGardeners@Work construct quality...,Fencing|Gardener|Lawn Mowing|Tree Lopping,Gardener
2,"Melbourne Computer And Business Consultants Hi, I offer services for home and small business com...",Computer Repair,Computer Repair
3,Looker Electrics Looker Electrics providing the very best electrical contracting services.\nI sp...,Electrician,Air Conditioning and Heating
4,Wire Not Electrical Pty Ltd Wire-Not Electrical is an awarding winning locally-owned and operate...,Electrician,Security System


In [13]:
stopwords = nltk.corpus.stopwords.words('english')
wn = nltk.WordNetLemmatizer()
ps = nltk.PorterStemmer()

In [7]:
features_importances_df=pd.read_csv('features_importances.csv')
import_features_list=list(features_importances_df['feature'])
len(import_features_list)

38077

In [5]:
bool(re.search('\d','a3bc'))

True

In [14]:
def clean_text_by_porter_stem(text):

    text = "".join([word.lower() for word in text if word not in string.punctuation])
    tokens = re.split('\W+', text)
    text = [ps.stem(word) for word in tokens if (word not in stopwords) and (word.isdigit() ==False)  ]
    #text = [ word for word in text if word in import_features_list]
    
    return text

def multiLabelBinarizer(text):
    if text is None: 
        return ['']
    categories = re.split('\|', text)
    return categories

def clean_text_by_wordnet_lemmatize(text):

    text = "".join([word.lower() for word in text if word not in string.punctuation])
    tokens = re.split('\W+', text)
    text = [wn.lemmatize(word)  for word in tokens if (word not in stopwords) and (word.isdigit() ==False)] 
    text = [ word for word in text if word in import_features_list]
    
    return text

In [13]:
#data.drop('feature_cleaned', axis=1, inplace=True)

In [28]:
data['feature_cleaned_ps']=data['feature'].apply(lambda x: clean_text_by_porter_stem( ('' if x is None else x).lower()))
data['feature_cleaned_wd']=data['feature'].apply(lambda x: clean_text_by_wordnet_lemmatize( ('' if x is None else x).lower()))

In [29]:
data.head()

,feature,label,feature_cleaned_ps,feature_cleaned_wd
0,"Anaxim Accounting Solutions The company is a qualified, professional accountant who deals with a...",Accountant,"[anaxim, account, solut, compani, qualifi, profession, account, deal, array, account, issu, prov...","[anaxim, accounting, solution, company, qualified, professional, accountant, deal, array, accoun..."
1,RCS Websites Let RCS Websites breath life in to your existing website!\r\nOur websites are simpl...,SEO and SEM,"[rc, websit, let, rc, websit, breath, life, exist, websit, websit, simpl, clean, profession, sur...","[rcs, website, let, rcs, website, breath, life, existing, website, website, simple, clean, profe..."
2,Intouch Carpet Cleaning Services InTouch Carpet and Pest Control is a family owned business who ...,Pest Control,"[intouch, carpet, clean, servic, intouch, carpet, pest, control, famili, own, busi, valu, old, f...","[intouch, carpet, cleaning, service, intouch, carpet, pest, control, family, owned, business, va..."
3,Shannon Thomas Carpenter Licensed Carpenter\r\n - Small Decks and Pergolas as well as re deck...,Carpenter,"[shannon, thoma, carpent, licens, carpent, small, deck, pergola, well, deck, repair, attent, det...","[shannon, thomas, carpenter, licensed, carpenter, small, deck, pergola, well, decking, repair, a..."
4,"M&M Cabinets Custom-made Cabinets for kitchen, laundries, bathroom and anywhere in your house. 3...",Builder,"[mm, cabinet, custommad, cabinet, kitchen, laundri, bathroom, anywher, hous, 3d, comput, draw, a...","[mm, cabinet, custommade, cabinet, kitchen, laundry, bathroom, anywhere, house, 3d, computer, dr..."


In [15]:
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer

In [16]:

count_vectorizer_categories = CountVectorizer(analyzer=multiLabelBinarizer)
count_vectorized_matrix_categories = count_vectorizer_categories.fit_transform(data['categories'] )

tfidf_vectorizer_ps = TfidfVectorizer(analyzer=clean_text_by_porter_stem)
tfidf_vectorized_matrix_ps = tfidf_vectorizer_ps.fit_transform(data['feature'] )



In [11]:
count_vectorizer_ps = CountVectorizer(analyzer=clean_text_by_porter_stem)
count_vectorized_matrix_ps = count_vectorizer_ps.fit_transform(data['feature'] )

In [17]:
count_vectorized_matrix_categories.shape

(39081, 139)

In [18]:
tfidf_vectorized_matrix_ps.shape

(39081, 48735)

In [19]:
from scipy.sparse import coo_matrix, hstack
import pandas as pd

print(pd.DataFrame(hstack([tfidf_vectorized_matrix_ps[0:10],count_vectorized_matrix_categories[0:10]], format='csr').toarray()))
type(hstack([tfidf_vectorized_matrix_ps[0:10],count_vectorized_matrix_categories[0:10]], format='csr'))

vectorized_matrix = hstack([tfidf_vectorized_matrix_ps,count_vectorized_matrix_categories], format='csr')

   0      1      2      3      4      5      6      7      8      9      \
0    0.0    0.0    0.0    0.0    0.0    0.0    0.0    0.0    0.0    0.0   
1    0.0    0.0    0.0    0.0    0.0    0.0    0.0    0.0    0.0    0.0   
2    0.0    0.0    0.0    0.0    0.0    0.0    0.0    0.0    0.0    0.0   
3    0.0    0.0    0.0    0.0    0.0    0.0    0.0    0.0    0.0    0.0   
4    0.0    0.0    0.0    0.0    0.0    0.0    0.0    0.0    0.0    0.0   
5    0.0    0.0    0.0    0.0    0.0    0.0    0.0    0.0    0.0    0.0   
6    0.0    0.0    0.0    0.0    0.0    0.0    0.0    0.0    0.0    0.0   
7    0.0    0.0    0.0    0.0    0.0    0.0    0.0    0.0    0.0    0.0   
8    0.0    0.0    0.0    0.0    0.0    0.0    0.0    0.0    0.0    0.0   
9    0.0    0.0    0.0    0.0    0.0    0.0    0.0    0.0    0.0    0.0   

   ...    48864  48865  48866  48867  48868  48869  48870  48871  48872  48873  
0  ...      0.0    0.0    0.0    0.0    0.0    0.0    0.0    0.0    0.0    0.0  
1  ...      

In [21]:
tfidf_vectorized_df_ps = pd.DataFrame(tfidf_vectorized_matrix_ps.toarray())

MemoryError: 

In [18]:
tfidf_vectorized_df_ps.head()

,0,1,2,3,4,5,6,7,8,9,...,48564,48565,48566,48567,48568,48569,48570,48571,48572,48573
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [9]:
tfidf_vectorizer_wn = TfidfVectorizer(analyzer=clean_text_by_wordnet_lemmatize)
tfidf_vectorized_matrix_wn = tfidf_vectorizer_wn.fit_transform(data['feature'])

In [10]:
tfidf_vectorized_matrix_wn.shape

(38974, 56825)

In [ ]:
tfidf_vectorizer_wn = TfidfVectorizer(analyzer=clean_text_by_wordnet_lemmatize)
tfidf_vectorized_matrix_wn = tfidf_vectorizer_wn.fit_transform(data['feature'])
tfidf_vectorized_df_wn = pd.DataFrame(tfidf_vectorized_matrix_wn.toarray())

In [19]:
tfidf_vectorized_df_ps_h = tfidf_vectorized_df_ps
tfidf_vectorized_df_ps_h.columns = tfidf_vectorizer_ps.get_feature_names()
tfidf_vectorized_df_ps_h.head()

,,00,000107l,000684l,000710l,001023l,01000sqm,0118651a01,012km,024481i,...,ўвўfrighten,ўвўhave,ўвўin,ўёўў,ўўв,中國理解,在墨尔本的中国婚礼乐队,ﬁnanc,ﬁnancial,ﬁngertip
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [20]:
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import precision_recall_fscore_support as score
import time

## Start to evaluate models

In [21]:
from sklearn.metrics import precision_recall_fscore_support as score
from sklearn.model_selection import train_test_split

In [22]:
X_train, X_test, y_train, y_test = train_test_split(vectorized_matrix, data['label'], test_size=0.2)

In [23]:
def train_RF(n_est, depth):
    rf = RandomForestClassifier(n_estimators=n_est, max_depth=depth, n_jobs=-1, verbose=10)
    rf_model = rf.fit(X_train, y_train)
    y_pred = rf_model.predict(X_test)
    precision, recall, fscore, support = score(y_test, y_pred)
    print('Est: {} / Depth: {} ---- Precision: {} / Recall: {} / Accuracy: {}'.format(
        n_est, depth, precision, recall,
        round((y_pred==y_test).sum() / len(y_pred), 3)))

In [24]:
from ipywidgets import FloatProgress
from IPython.display import display

In [25]:
max_count = 12
count = 0
progressBar = FloatProgress(min=0, max=max_count)
display(progressBar)

for n_est in [50, 10, 300]:
    for depth in [10, 20, 30, None]:
        print("\n\nTraining combination: n_est:{n_est} - depth:{depth}".format(n_est=str(n_est),depth=str(depth)))
        progressBar.value+=1
        count += 1
        train_RF(n_est, depth)

A Jupyter Widget



Training combination: n_est:50 - depth:10
building tree 1 of 50
building tree 2 of 50building tree 4 of 50
building tree 3 of 50

building tree 5 of 50
building tree 6 of 50
building tree 7 of 50
building tree 8 of 50
building tree 9 of 50
building tree 10 of 50
building tree 11 of 50
building tree 12 of 50
building tree 13 of 50


[Parallel(n_jobs=-1)]: Done   5 tasks      | elapsed:    0.3s
[Parallel(n_jobs=-1)]: Done  10 tasks      | elapsed:    0.5s


building tree 14 of 50
building tree 15 of 50
building tree 16 of 50
building tree 17 of 50
building tree 18 of 50
building tree 19 of 50
building tree 20 of 50
building tree 21 of 50
building tree 22 of 50
building tree 23 of 50

[Parallel(n_jobs=-1)]: Done  17 tasks      | elapsed:    0.8s


...

building tree 262 of 300
building tree 263 of 300
building tree 264 of 300
building tree 265 of 300
building tree 266 of 300
building tree 267 of 300
building tree 268 of 300
building tree 269 of 300
building tree 270 of 300
building tree 271 of 300
building tree 272 of 300
building tree 273 of 300
building tree 274 of 300
building tree 275 of 300
building tree 276 of 300
building tree 277 of 300
building tree 278 of 300
building tree 279 of 300
building tree 280 of 300
building tree 281 of 300
building tree 282 of 300
building tree 283 of 300
building tree 284 of 300


[Parallel(n_jobs=-1)]: Done 280 tasks      | elapsed:  8.5min


building tree 285 of 300
building tree 286 of 300
building tree 287 of 300
building tree 288 of 300
building tree 289 of 300
building tree 290 of 300
building tree 291 of 300
building tree 292 of 300
building tree 293 of 300
building tree 294 of 300
building tree 295 of 300
building tree 296 of 300
building tree 297 of 300
building tree 298 of 300
building tree 299 of 300
building tree 300 of 300


[Parallel(n_jobs=-1)]: Done 300 out of 300 | elapsed:  9.0min finished
[Parallel(n_jobs=4)]: Done   5 tasks      | elapsed:    0.5s
[Parallel(n_jobs=4)]: Done  10 tasks      | elapsed:    1.0s
[Parallel(n_jobs=4)]: Done  17 tasks      | elapsed:    2.7s
[Parallel(n_jobs=4)]: Done  24 tasks      | elapsed:    2.9s
[Parallel(n_jobs=4)]: Done  33 tasks      | elapsed:    3.0s
[Parallel(n_jobs=4)]: Done  42 tasks      | elapsed:    3.2s
[Parallel(n_jobs=4)]: Done  53 tasks      | elapsed:    3.4s
[Parallel(n_jobs=4)]: Done  64 tasks      | elapsed:    4.1s
[Parallel(n_jobs=4)]: Done  77 tasks      | elapsed:    6.5s
[Parallel(n_jobs=4)]: Done  90 tasks      | elapsed:    8.4s
[Parallel(n_jobs=4)]: Done 105 tasks      | elapsed:    9.0s
[Parallel(n_jobs=4)]: Done 120 tasks      | elapsed:    9.3s
[Parallel(n_jobs=4)]: Done 137 tasks      | elapsed:    9.6s
[Parallel(n_jobs=4)]: Done 154 tasks      | elapsed:   12.8s
[Parallel(n_jobs=4)]: Done 173 tasks      | elapsed:   13.2s
[Parallel(n_jo

Est: 300 / Depth: None ---- Precision: [ 0.96010638  0.88        0.76851852  0.78947368  0.8125      0.57142857
  0.71428571  1.          0.          0.625       0.8         0.55247286
  0.          0.76470588  0.          0.          0.98113208  1.          0.125
  0.76666667  0.94871795  0.          0.94029851  0.94736842  0.62826421
  0.83431953  0.7027027   0.76923077  0.82608696  0.25        0.78571429
  0.34911243  0.5         0.90909091  0.73809524  0.75        0.6         1.
  0.67924528  0.57142857  0.5         0.          0.5641791   0.
  0.92682927  0.6125      0.72727273  0.28205128  0.76923077  0.2         0.5
  0.37078652  1.          0.57758621  0.          0.71428571  0.88888889
  0.48701299  0.83333333  0.808       1.          0.92307692  1.
  0.42857143  1.          0.27932961  0.66666667  0.9047619   0.
  0.97802198  0.95876289  0.71428571  0.89655172  0.8         0.68454259
  1.          0.55555556  0.8343949   0.66666667  0.8         0.
  0.66666667  0.86363636  0.

In [31]:
train_RF(900, None) # 0.613 accuracy if i use wordnet lemamtizing

building tree 1 of 900
building tree 4 of 900
building tree 3 of 900
building tree 2 of 900
building tree 5 of 900
building tree 6 of 900
building tree 7 of 900
building tree 8 of 900


[Parallel(n_jobs=-1)]: Done   5 tasks      | elapsed:   11.2s


building tree 9 of 900
building tree 10 of 900
building tree 11 of 900
building tree 12 of 900
building tree 13 of 900
building tree 14 of 900
building tree 15 of 900
building tree 16 of 900

[Parallel(n_jobs=-1)]: Done  10 tasks      | elapsed:   17.6s



building tree 17 of 900
building tree 18 of 900
building tree 19 of 900
building tree 20 of 900
building tree 21 of 900


...

building tree 837 of 900
building tree 838 of 900
building tree 839 of 900
building tree 840 of 900
building tree 841 of 900
building tree 842 of 900
building tree 843 of 900
building tree 844 of 900
building tree 845 of 900
building tree 846 of 900
building tree 847 of 900
building tree 848 of 900
building tree 849 of 900
building tree 850 of 900
building tree 851 of 900
building tree 852 of 900
building tree 853 of 900
building tree 854 of 900
building tree 855 of 900
building tree 856 of 900
building tree 857 of 900
building tree 858 of 900
building tree 859 of 900
building tree 860 of 900
building tree 861 of 900
building tree 862 of 900
building tree 863 of 900
building tree 864 of 900
building tree 865 of 900
building tree 866 of 900
building tree 867 of 900
building tree 868 of 900
building tree 869 of 900
building tree 870 of 900
building tree 871 of 900
building tree 872 of 900
building tree 873 of 900
building tree 874 of 900
building tree 875 of 900
building tree 876 of 900


[Parallel(n_jobs=-1)]: Done 874 tasks      | elapsed: 28.2min


building tree 878 of 900
building tree 879 of 900
building tree 880 of 900
building tree 881 of 900
building tree 882 of 900
building tree 883 of 900
building tree 884 of 900
building tree 885 of 900
building tree 886 of 900
building tree 887 of 900
building tree 888 of 900
building tree 889 of 900
building tree 890 of 900
building tree 891 of 900
building tree 892 of 900
building tree 893 of 900
building tree 894 of 900
building tree 895 of 900
building tree 896 of 900
building tree 897 of 900
building tree 898 of 900
building tree 899 of 900
building tree 900 of 900


[Parallel(n_jobs=-1)]: Done 900 out of 900 | elapsed: 28.9min finished
[Parallel(n_jobs=4)]: Done   5 tasks      | elapsed:    0.6s
[Parallel(n_jobs=4)]: Done  10 tasks      | elapsed:    1.3s
[Parallel(n_jobs=4)]: Done  17 tasks      | elapsed:    1.8s
[Parallel(n_jobs=4)]: Done  24 tasks      | elapsed:    1.9s
[Parallel(n_jobs=4)]: Done  33 tasks      | elapsed:    2.4s
[Parallel(n_jobs=4)]: Done  42 tasks      | elapsed:    3.1s
[Parallel(n_jobs=4)]: Done  53 tasks      | elapsed:    6.5s
[Parallel(n_jobs=4)]: Done  64 tasks      | elapsed:    6.9s
[Parallel(n_jobs=4)]: Done  77 tasks      | elapsed:    7.2s
[Parallel(n_jobs=4)]: Done  90 tasks      | elapsed:    7.4s
[Parallel(n_jobs=4)]: Done 105 tasks      | elapsed:    7.8s
[Parallel(n_jobs=4)]: Done 120 tasks      | elapsed:   11.0s
[Parallel(n_jobs=4)]: Done 137 tasks      | elapsed:   11.9s
[Parallel(n_jobs=4)]: Done 154 tasks      | elapsed:   12.3s
[Parallel(n_jobs=4)]: Done 173 tasks      | elapsed:   12.8s
[Parallel(n_jo

Est: 900 / Depth: None ---- Precision: [ 0.8617284   0.77777778  0.73214286  0.6         0.47826087  0.53731343
  0.6875      1.          0.          0.          0.75862069  0.45292621
  0.          0.73333333  0.          0.          0.89830508  0.375       0.15
  0.61333333  0.84313725  0.          0.86861314  0.875       0.5744089
  0.81656805  0.62264151  0.85106383  0.85714286  0.          0.79487179
  0.21052632  0.44444444  0.75862069  0.39285714  0.63636364  0.35534591
  1.          0.7826087   0.66666667  0.          0.          0.53250774
  0.          0.83333333  0.62765957  0.57142857  0.28        0.6
  0.22222222  0.          0.47142857  0.6875      0.38848921  0.
  0.57894737  0.66666667  0.51369863  0.8         0.62328767  1.          1.
  0.          0.37647059  0.8         0.12        0.          0.63636364
  0.          0.90116279  0.92682927  1.          0.91025641  0.52173913
  0.65373134  0.          0.21568627  0.77925532  0.          0.36363636
  0.          0.64

In [31]:
train_RF(300, None) # 0.618 accuracy if i use wordnet lemamtizing

building tree 1 of 300
building tree 2 of 300building tree 3 of 300building tree 4 of 300


building tree 5 of 300
building tree 6 of 300
building tree 7 of 300
building tree 8 of 300


[Parallel(n_jobs=-1)]: Done   5 tasks      | elapsed:   13.0s


building tree 9 of 300
building tree 10 of 300
building tree 11 of 300
building tree 12 of 300
building tree 13 of 300
building tree 14 of 300
building tree 15 of 300


[Parallel(n_jobs=-1)]: Done  10 tasks      | elapsed:   20.8s


building tree 16 of 300
building tree 17 of 300
building tree 18 of 300
building tree 19 of 300
building tree 20 of 300
building tree 21 of 300


...

building tree 261 of 300
building tree 262 of 300
building tree 263 of 300
building tree 264 of 300
building tree 265 of 300
building tree 266 of 300
building tree 267 of 300
building tree 268 of 300
building tree 269 of 300
building tree 270 of 300
building tree 271 of 300
building tree 272 of 300
building tree 273 of 300
building tree 274 of 300
building tree 275 of 300
building tree 276 of 300
building tree 277 of 300
building tree 278 of 300
building tree 279 of 300
building tree 280 of 300
building tree 281 of 300
building tree 282 of 300
building tree 283 of 300
building tree 284 of 300


[Parallel(n_jobs=-1)]: Done 280 tasks      | elapsed:  8.3min


building tree 285 of 300
building tree 286 of 300
building tree 287 of 300
building tree 288 of 300
building tree 289 of 300
building tree 290 of 300
building tree 291 of 300
building tree 292 of 300
building tree 293 of 300
building tree 294 of 300
building tree 295 of 300
building tree 296 of 300
building tree 297 of 300
building tree 298 of 300
building tree 299 of 300
building tree 300 of 300


[Parallel(n_jobs=-1)]: Done 300 out of 300 | elapsed:  8.7min finished
[Parallel(n_jobs=4)]: Done   5 tasks      | elapsed:    0.1s
[Parallel(n_jobs=4)]: Done  10 tasks      | elapsed:    0.6s
[Parallel(n_jobs=4)]: Done  17 tasks      | elapsed:    1.4s
[Parallel(n_jobs=4)]: Done  24 tasks      | elapsed:    1.6s
[Parallel(n_jobs=4)]: Done  33 tasks      | elapsed:    2.2s
[Parallel(n_jobs=4)]: Done  42 tasks      | elapsed:    2.9s
[Parallel(n_jobs=4)]: Done  53 tasks      | elapsed:    5.2s
[Parallel(n_jobs=4)]: Done  64 tasks      | elapsed:    7.4s
[Parallel(n_jobs=4)]: Done  77 tasks      | elapsed:    8.4s
[Parallel(n_jobs=4)]: Done  90 tasks      | elapsed:    8.7s
[Parallel(n_jobs=4)]: Done 105 tasks      | elapsed:    9.0s
[Parallel(n_jobs=4)]: Done 120 tasks      | elapsed:    9.9s
[Parallel(n_jobs=4)]: Done 137 tasks      | elapsed:   15.0s
[Parallel(n_jobs=4)]: Done 154 tasks      | elapsed:   16.1s
[Parallel(n_jobs=4)]: Done 173 tasks      | elapsed:   17.7s
[Parallel(n_jo

Est: 300 / Depth: None ---- Precision: [ 0.90364583  0.85714286  0.61344538  0.65        0.46        0.52542373
  0.8125      0.90909091  0.          0.33333333  0.68656716  0.45974026
  0.          0.66666667  0.          0.          0.890625    0.85714286
  0.          0.71830986  0.79518072  0.          0.88429752  0.88235294
  0.60583942  0.80851064  0.70050761  0.87037037  0.84615385  0.          0.75
  0.0962963   0.33333333  0.77777778  0.42307692  0.67741935  0.34726688
  1.          0.78        0.56        0.          0.          0.55214724
  0.          0.8125      0.54954955  0.44444444  0.34285714  0.33333333
  0.25        0.          0.42857143  0.95454545  0.38562092  0.          0.6
  0.75        0.44654088  0.83333333  0.73228346  0.71428571  0.66666667
  0.          0.32361516  0.86666667  0.19230769  1.          0.55555556
  0.          0.9068323   0.88429752  1.          0.86206897  0.47368421
  0.73538462  1.          0.36956522  0.80678851  0.          0.375       

In [14]:
from sklearn.model_selection import GridSearchCV

rf = RandomForestClassifier()
param = {'n_estimators': [10, 150, 300],
        'max_depth': [30, 90, None],
         'verbose': [10]
        }


In [ ]:
gs = GridSearchCV(rf, param, cv=5, n_jobs=-1, verbose=10)
gs_fit = gs.fit(tfidf_vectorized_matrix_ps, data['label'])
pd.DataFrame(gs_fit.cv_results_).sort_values('mean_test_score', ascending=False)[0:5]

Fitting 5 folds for each of 9 candidates, totalling 45 fits


C:\Users\chrisq\AppData\Local\Continuum\anaconda3\lib\site-packages\sklearn\model_selection\_split.py:597: Warning: The least populated class in y has only 3 members, which is too few. The minimum number of members in any class cannot be less than n_splits=5.
  % (min_groups, self.n_splits)), Warning)
[Parallel(n_jobs=-1)]: Done   5 tasks      | elapsed:   17.2s
[Parallel(n_jobs=-1)]: Done  10 tasks      | elapsed:  4.2min
[Parallel(n_jobs=-1)]: Done  17 tasks      | elapsed:  8.6min
[Parallel(n_jobs=-1)]: Done  24 tasks      | elapsed: 21.6min


### my laptop doesnt have enough memory to run GridsearchCV above 
### so I ran the same on EC2 instance and got n_est 300 and depth None as best mean test score

In [15]:
def train_GB(est, max_depth, lr):
    gb = GradientBoostingClassifier(n_estimators=est, max_depth=max_depth, learning_rate=lr , verbose=10)
    gb_model = gb.fit(X_train, y_train)
    y_pred = gb_model.predict(X_test)
    precision, recall, fscore, train_support = score(y_test, y_pred)
    print('Est: {} / Depth: {} / LR: {} ---- Precision: {} / Recall: {} / Accuracy: {}'.format(
        est, max_depth, lr, precision,recall, 
        round((y_pred==y_test).sum()/len(y_pred), 3)))

In [16]:
from ipywidgets import FloatProgress
from IPython.display import display

In [ ]:
max_count = 12
count = 0
progressBar = FloatProgress(min=0, max=max_count)
display(progressBar)

for n_est in [50, 100, 150]:
    for max_depth in [3, 7, 11, 15]:
        print("\n\nTraining combination: n_est:{n_est} - max_depth:{max_depth}".format(n_est=str(n_est),max_depth=str(max_depth)))
        progressBar.value+=1
        count += 1
        train_GB(n_est, max_depth, 0.01)

A Jupyter Widget



Training combination: 50 - 3


C:\Users\chrisq\AppData\Local\Continuum\anaconda3\lib\site-packages\sklearn\metrics\classification.py:1135: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples.
  'precision', 'predicted', average, warn_for)
C:\Users\chrisq\AppData\Local\Continuum\anaconda3\lib\site-packages\sklearn\metrics\classification.py:1137: UndefinedMetricWarning: Recall and F-score are ill-defined and being set to 0.0 in labels with no true samples.
  'recall', 'true', average, warn_for)


Est: 50 / Depth: 3 / LR: 0.01 ---- Precision: [ 0.93452381  0.96        0.73451327  0.75        0.28        0.48        0.71875
  0.45        0.          0.41666667  0.81538462  0.34558824  0.
  0.72972973  0.          0.16666667  0.74545455  0.83333333  0.07692308
  0.61458333  0.83333333  0.          0.90833333  0.75        0.65492958
  0.81761006  0.7         0.69565217  0.76470588  0.5         0.81428571
  0.30508475  0.57142857  0.90243902  0.53125     0.53846154  0.28851541
  0.66666667  0.65137615  0.38461538  0.83333333  1.          0.49832776
  0.          0.86956522  0.43434343  0.66666667  0.19354839  0.66666667
  0.08333333  0.55555556  0.40243902  1.          0.4159292   0.          0.53125
  1.          0.55555556  0.81578947  0.8         0.53846154  0.76470588
  0.          0.36394558  0.84615385  0.11363636  0.75        0.59259259
  0.          0.91156463  0.8877551   0.          0.89041096  0.41176471
  0.68543046  0.625       0.18681319  0.77777778  0.          0.6666

Est: 50 / Depth: 15 / LR: 0.01 ---- Precision: [ 0.94864048  0.95238095  0.62601626  0.59375     0.34285714  0.49019608
  0.64        0.64285714  0.          0.          0.77777778  0.27158099
  0.          0.64516129  0.          0.2         0.84444444  0.77777778
  0.03846154  0.52991453  0.79710145  0.          0.90677966  0.8
  0.66793893  0.82550336  0.69871795  0.70212766  0.92857143  0.
  0.81355932  0.17682927  0.5         0.92105263  0.48275862  0.46666667
  0.31481481  0.42857143  0.6         0.30434783  0.83333333  1.          0.4924812
  0.          0.82978723  0.44615385  0.52941176  0.09174312  0.42307692
  0.25        0.14814815  0.32467532  1.          0.37162162  0.
  0.51612903  1.          0.56521739  0.8358209   0.77777778  0.4
  0.90909091  0.          0.37089202  0.8         0.08823529  0.5
  0.45714286  0.07692308  0.93377483  0.91304348  0.25        0.92647059
  0.44444444  0.67054264  0.71428571  0.15789474  0.78456592  0.          0.6
  0.          0.37209302 

In [ ]:
gb = GradientBoostingClassifier()
param = {
    'n_estimators': [50, 100, 150], 
    'max_depth': [7, 11, 15],
    'learning_rate': [0.01]
}

clf = GridSearchCV(gb, param, cv=5, n_jobs=-1)
cv_fit = clf.fit(tfidf_vectorized_matrix_ps, data['label'])
pd.DataFrame(cv_fit.cv_results_).sort_values('mean_test_score', ascending=False)[0:5]

### start to predict

In [26]:
tfidf_vectorizer_ps = TfidfVectorizer(analyzer=clean_text_by_porter_stem)
count_vectorizer_categories = CountVectorizer(analyzer=multiLabelBinarizer)

tfidf_vect_words_fit = tfidf_vectorizer_ps.fit(data['feature'])
count_vect_categories_fit = count_vectorizer_categories.fit(data['categories'])

tfidf_words_train = tfidf_vect_words_fit.transform(data['feature'])
count_categories_train = count_vect_categories_fit.transform(data['categories'])

vectorized_matrix_train = hstack([tfidf_words_train,count_categories_train], format='csr')

In [27]:
rf = RandomForestClassifier(n_estimators=300, max_depth=None, n_jobs=-1, verbose=10)
rf_model = rf.fit(vectorized_matrix_train, data['label'])

building tree 1 of 300
building tree 4 of 300building tree 3 of 300building tree 2 of 300


building tree 5 of 300
building tree 6 of 300
building tree 7 of 300
building tree 8 of 300


[Parallel(n_jobs=-1)]: Done   5 tasks      | elapsed:   13.4s


building tree 9 of 300
building tree 10 of 300
building tree 11 of 300
building tree 12 of 300
building tree 13 of 300


[Parallel(n_jobs=-1)]: Done  10 tasks      | elapsed:   21.3s


building tree 14 of 300
building tree 15 of 300
building tree 16 of 300
building tree 17 of 300
building tree 18 of 300
building tree 19 of 300
building tree 20 of 300
building tree 21 of 300


...

[Parallel(n_jobs=-1)]: Done 257 tasks      | elapsed:  7.5min


building tree 261 of 300
building tree 262 of 300
building tree 263 of 300
building tree 264 of 300
building tree 265 of 300
building tree 266 of 300
building tree 267 of 300
building tree 268 of 300
building tree 269 of 300
building tree 270 of 300
building tree 271 of 300
building tree 272 of 300
building tree 273 of 300
building tree 274 of 300
building tree 275 of 300
building tree 276 of 300
building tree 277 of 300
building tree 278 of 300
building tree 279 of 300
building tree 280 of 300
building tree 281 of 300
building tree 282 of 300
building tree 283 of 300
building tree 284 of 300


[Parallel(n_jobs=-1)]: Done 280 tasks      | elapsed:  8.1min


building tree 285 of 300
building tree 286 of 300
building tree 287 of 300
building tree 288 of 300
building tree 289 of 300
building tree 290 of 300
building tree 291 of 300
building tree 292 of 300
building tree 293 of 300
building tree 294 of 300
building tree 295 of 300
building tree 296 of 300
building tree 297 of 300
building tree 298 of 300
building tree 299 of 300
building tree 300 of 300


[Parallel(n_jobs=-1)]: Done 300 out of 300 | elapsed:  8.7min finished


In [44]:
features_importances_df=pd.DataFrame(list(zip(tfidf_vectorizer_ps.get_feature_names() + count_vectorizer_categories.get_feature_names(),rf_model.feature_importances_)))
features_importances_df.columns = ['feature','importance']


,feature,importance
48830,Plumber,0.006462
33061,plumb,0.006246
9168,clean,0.006104
38486,servic,0.005954
48837,Removalist,0.005578
48745,Bathroom Renovations,0.005392
48748,Bookkeeping,0.005279
1695,account,0.005231
48764,Cleaner,0.005099
48736,Accountant,0.004665


In [ ]:
features_importances_df.sort_values('importance',ascending =False)

In [28]:
QUERY = """ 

with business_categories as (
select  bc.business_detail_id
,listagg(c.name,'|') within group (order by c.name) as categories
from mysql_oneflare_production.business_categories bc 
left join mysql_oneflare_production.categories c on bc.category_id = c.id
group by 1
)


SELECT
bm.id
,trim(nvl(bm.name,'') || ' ' || nvl(description,'')) "feature"
,bc.categories
FROM oneflare_reports.business_master bm
left join business_categories bc on bm.id = bc.business_detail_id
WHERE bm.total_quotes_made < 3
"""

In [29]:
data_to_predict=pd.read_sql(QUERY,redshift)

In [30]:
data_to_predict.head()

,id,feature,categories
0,54,Flushing Success,Carpet Cleaning|Cleaner|Grout Cleaner|Office Cleaning|Pool Cleaner
1,292,Yarra Plenty Accountants Yarra Plenty Accountants can help with all your bookkeeping requirement...,Accountant|Bookkeeping|Tax Accountant
2,645,Adrian Gold Painter Adrian Gold Painter has been in the painting business for many of our client...,Painter|Printing|Signwriter|T-Shirt Printing
3,657,Ads Painting pty ltd I provide a first class painting service for your home or business.\nWe ha...,Painter
4,709,Cutting Edge Painting,Interior Design|Painter|Wallpaper


In [31]:
#tfidf_vectorizer_ps = TfidfVectorizer(analyzer=clean_text_by_porter_stem)
#tfidf_vectorized_matrix_to_predict_ps = tfidf_vect_fit.transform(data_to_predict['feature'] )

tfidf_words_to_predict = tfidf_vect_words_fit.transform(data_to_predict['feature'])
count_categories_to_predict = count_vect_categories_fit.transform(data_to_predict['categories'])

vectorized_matrix_to_predict = hstack([tfidf_words_to_predict,count_categories_to_predict], format='csr')


In [32]:
y_to_predict = rf_model.predict(vectorized_matrix_to_predict)

[Parallel(n_jobs=4)]: Done   5 tasks      | elapsed:    0.3s
[Parallel(n_jobs=4)]: Done  10 tasks      | elapsed:    0.6s
[Parallel(n_jobs=4)]: Done  17 tasks      | elapsed:    1.1s
[Parallel(n_jobs=4)]: Done  24 tasks      | elapsed:    1.4s
[Parallel(n_jobs=4)]: Done  33 tasks      | elapsed:    2.2s
[Parallel(n_jobs=4)]: Done  42 tasks      | elapsed:    6.3s
[Parallel(n_jobs=4)]: Done  53 tasks      | elapsed:    7.0s
[Parallel(n_jobs=4)]: Done  64 tasks      | elapsed:    7.6s
[Parallel(n_jobs=4)]: Done  77 tasks      | elapsed:    9.9s
[Parallel(n_jobs=4)]: Done  90 tasks      | elapsed:   10.9s
[Parallel(n_jobs=4)]: Done 105 tasks      | elapsed:   14.7s
[Parallel(n_jobs=4)]: Done 120 tasks      | elapsed:   15.4s
[Parallel(n_jobs=4)]: Done 137 tasks      | elapsed:   17.9s
[Parallel(n_jobs=4)]: Done 154 tasks      | elapsed:   18.7s
[Parallel(n_jobs=4)]: Done 173 tasks      | elapsed:   21.2s
[Parallel(n_jobs=4)]: Done 192 tasks      | elapsed:   23.5s
[Parallel(n_jobs=4)]: Do

In [34]:
data_to_predict['predited_category'] = y_to_predict

In [35]:
data_to_predict.to_csv('predited_category_rf.csv',index=False)

In [36]:
pd.set_option('display.max_colwidth', -1)
data_to_predict.head()

,id,feature,categories,predited_category
0,54,Flushing Success,Carpet Cleaning|Cleaner|Grout Cleaner|Office Cleaning|Pool Cleaner,Cleaner
1,292,"Yarra Plenty Accountants Yarra Plenty Accountants can help with all your bookkeeping requirements.\nI have a Cert IV in Accounting.\nSpecialising in MYOB, Reckon and Quickbooks, Yarra Plenty Accountants provide a range of services to individuals and small businesses.",Accountant|Bookkeeping|Tax Accountant,Accountant
2,645,Adrian Gold Painter Adrian Gold Painter has been in the painting business for many of our clients.\nWe have a high attention to detail and pride in our work.\nReliable and professional.,Painter|Printing|Signwriter|T-Shirt Printing,Painter
3,657,"Ads Painting pty ltd I provide a first class painting service for your home or business.\nWe have been in the painting and decorating business for many years.\nThe company are highly skilled, professional and extremely reliable providing the very best painting services.",Painter,Painter
4,709,Cutting Edge Painting,Interior Design|Painter|Wallpaper,Partitioning


In [32]:
from sklearn.externals import joblib
joblib.dump(rf_model, 'random_forest.pkl') 
#clf = joblib.load('filename.pk1')

['random_forest.pkl']